# 🌾 AgriDash — Pemodelan Data Tanaman Pangan
### Kabupaten Kediri | 2020–2025

| Bagian | Isi |
|--------|-----|
| 1 | Setup & Konfigurasi |
| 2 | EDA (Exploratory Data Analysis) |
| 3 | Clustering Kecamatan |
| 4 | Time-Series Forecast |
| 5 | Rekomendasi Tanaman |
| 6 | Export CSV + PKL (Deploy Bundle) |
| 7 | Ringkasan |

> **Catatan**: Notebook ini menghasilkan satu file `agridash_model_bundle.pkl` yang berisi
> seluruh artefak model, data forecast, dan metadata — siap di-load oleh API backend.

## 1. Setup & Konfigurasi

In [ ]:
import subprocess, sys

PACKAGES = [
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'scikit-learn', 'statsmodels', 'scipy'
]
for pkg in PACKAGES:
    try:
        __import__(pkg if pkg != 'scikit-learn' else 'sklearn')
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print("✅ Semua library tersedia")

In [ ]:
import os, pickle, hashlib, json, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import linregress

warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
plt.rcParams.update({
    'figure.figsize'   : (12, 5),
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})
sns.set_style('whitegrid')
sns.set_palette('Set2')
print("✅ Import berhasil")

In [ ]:
# ── PATH KONFIGURASI ──────────────────────────────────────────
# Sesuaikan DATA_PATH dengan lokasi file CSV Anda
DATA_PATH  = '../data/raw/tanaman_pangan/Tanaman_Pangan_per_kecamatan.csv'
OUTPUT_DIR = '../output/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data    : {os.path.abspath(DATA_PATH)}")
print(f"Output  : {os.path.abspath(OUTPUT_DIR)}")

## 2. EDA (Exploratory Data Analysis)

In [ ]:
# ── 2.1 Load & Validasi Dataset ───────────────────────────────
df = pd.read_csv(DATA_PATH)

REQUIRED = ['Tahun','kecamatan','tanaman','luas_panen_ha','produksi_ton','produktivitas_kuintal_ha']
missing_cols = [c for c in REQUIRED if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

# Tipe data
for col in ['luas_panen_ha','produksi_ton','produktivitas_kuintal_ha','Tahun']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['kecamatan'] = df['kecamatan'].astype(str).str.strip()
df['tanaman']   = df['tanaman'].astype(str).str.strip()
df['Tahun']     = df['Tahun'].astype(int)

# Cek nilai kosong
n_null = df.isnull().sum().sum()
if n_null > 0:
    print(f"⚠️  Ditemukan {n_null} nilai kosong:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print("✅ Tidak ada nilai kosong")

# Cek duplikat
n_dup = df.duplicated(['Tahun','kecamatan','tanaman']).sum()
print(f"{'⚠️ ' if n_dup else '✅'} Duplikat baris: {n_dup}")

print(f"\n{'='*55}")
print(f"  PROFIL DATASET")
print(f"{'='*55}")
print(f"  Baris          : {len(df):,}")
print(f"  Kolom          : {df.shape[1]}")
print(f"  Rentang tahun  : {df['Tahun'].min()} – {df['Tahun'].max()}")
print(f"  Jumlah kecamatan: {df['kecamatan'].nunique()}")
print(f"  Jenis tanaman  : {df['tanaman'].nunique()}")
print(f"{'='*55}")
print(f"\nList tanaman: {sorted(df['tanaman'].unique().tolist())}")
df.head(8)

In [ ]:
# ── 2.2 Statistik Deskriptif per Tanaman ─────────────────────
for t in sorted(df['tanaman'].unique()):
    sub = df[df['tanaman'] == t]
    print(f"\n{'─'*55}")
    print(f"  {t.upper()}")
    print(f"{'─'*55}")
    desc = sub[['luas_panen_ha','produksi_ton','produktivitas_kuintal_ha']].describe().round(2)
    print(desc)

In [ ]:
# ── 2.3 Visualisasi — Produksi & Produktivitas per Tanaman ──
tanaman_list   = sorted(df['tanaman'].unique())
kecamatan_list = sorted(df['kecamatan'].unique())
colors = sns.color_palette('Set2', len(tanaman_list))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prod_tan = df.groupby('tanaman')['produksi_ton'].sum().sort_values(ascending=False)
bars = axes[0].bar(prod_tan.index, prod_tan.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Total Produksi per Jenis Tanaman (2020–2025)')
axes[0].set_xlabel('Jenis Tanaman')
axes[0].set_ylabel('Produksi (Ton)')
axes[0].tick_params(axis='x', rotation=15)
for bar, v in zip(bars, prod_tan.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + prod_tan.max()*0.01,
                 f'{v:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

prod_v = df.groupby('tanaman')['produktivitas_kuintal_ha'].mean().sort_values(ascending=False)
bars2  = axes[1].bar(prod_v.index, prod_v.values, color=colors[::-1], edgecolor='white', linewidth=0.8)
axes[1].set_title('Rata-rata Produktivitas per Jenis Tanaman')
axes[1].set_xlabel('Jenis Tanaman')
axes[1].set_ylabel('Produktivitas (Kw/Ha)')
axes[1].tick_params(axis='x', rotation=15)
for bar, v in zip(bars2, prod_v.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + prod_v.max()*0.01,
                 f'{v:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Overview Komoditas Tanaman Pangan — Kabupaten Kediri', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_01_produksi_per_tanaman.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ eda_01_produksi_per_tanaman.png")

In [ ]:
# ── 2.4 Tren Tahunan per Tanaman ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = [
    ('luas_panen_ha',           'Luas Panen (Ha)'),
    ('produksi_ton',            'Produksi (Ton)'),
    ('produktivitas_kuintal_ha','Produktivitas (Kw/Ha)'),
]

for ax, (col, label) in zip(axes, metrics):
    for i, t in enumerate(tanaman_list):
        sub = df[df['tanaman'] == t].groupby('Tahun')[col].sum()
        ax.plot(sub.index, sub.values, marker='o', label=t,
                color=colors[i], linewidth=2.2, markersize=7)
    ax.set_title(f'Tren {label}')
    ax.set_xlabel('Tahun')
    ax.set_ylabel(label)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.suptitle('Tren Tahunan Komoditas Tanaman Pangan (2020–2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_02_trend_tahunan.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ eda_02_trend_tahunan.png")

In [ ]:
# ── 2.5 Heatmap Produksi rata-rata per Kecamatan ─────────────
pivot_hm = df.groupby(['kecamatan','tanaman'])['produksi_ton'].mean().unstack(fill_value=0).round(0)

fig, ax = plt.subplots(figsize=(11, 11))
sns.heatmap(pivot_hm, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, linecolor='white', ax=ax,
            cbar_kws={'label': 'Produksi rata-rata (Ton/tahun)', 'shrink': 0.8})
ax.set_title('Heatmap Rata-rata Produksi per Kecamatan & Tanaman (2020–2025)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Jenis Tanaman')
ax.set_ylabel('Kecamatan')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_03_heatmap_kecamatan.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ eda_03_heatmap_kecamatan.png")

In [ ]:
# ── 2.6 Ranking Kecamatan berdasarkan Total Produksi ─────────
top_kec   = df.groupby('kecamatan')['produksi_ton'].sum().sort_values(ascending=True)
color_arr = ['#2ecc71' if v >= top_kec.quantile(0.75) else
             '#3498db' if v >= top_kec.quantile(0.50) else
             '#e67e22' if v >= top_kec.quantile(0.25) else '#e74c3c'
             for v in top_kec.values]

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(top_kec.index, top_kec.values, color=color_arr, edgecolor='white', height=0.7)
ax.set_title('Total Produksi per Kecamatan — Semua Tanaman Pangan (2020–2025)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Total Produksi (Ton)')
for bar, v in zip(bars, top_kec.values):
    ax.text(bar.get_width() + top_kec.max()*0.005,
            bar.get_y() + bar.get_height()/2,
            f'{v:,.0f}', va='center', fontsize=8.5)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Kuartil 4 (Tertinggi)'),
    Patch(facecolor='#3498db', label='Kuartil 3'),
    Patch(facecolor='#e67e22', label='Kuartil 2'),
    Patch(facecolor='#e74c3c', label='Kuartil 1 (Terendah)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_04_top_kecamatan.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ eda_04_top_kecamatan.png")

## 3. Clustering Kecamatan

**Tujuan**: Mengelompokkan kecamatan berdasarkan pola produksi relatif antar tanaman.

**Perbaikan dari versi sebelumnya**:
- Fitur = **proporsi luas panen** + **log1p(produksi)** → menghindari dominasi skala kedelai vs padi
- Label cluster ditentukan **otomatis** dari rata-rata produksi total di tiap centroid
- K optimal dipilih dari **silhouette tertinggi**, bukan hardcode

In [ ]:
# ── 3.1 Persiapan Feature Matrix ─────────────────────────────
# Fitur 1: rata-rata log1p(produksi) → meredam efek outlier skala besar
pivot_log = df.groupby(['kecamatan','tanaman'])['produksi_ton']               .mean().apply(np.log1p).unstack(fill_value=0)

# Fitur 2: proporsi luas panen per tanaman terhadap total luas kecamatan
pivot_luas = df.groupby(['kecamatan','tanaman'])['luas_panen_ha'].mean().unstack(fill_value=0)
total_luas  = pivot_luas.sum(axis=1).replace(0, np.nan)
pivot_prop  = pivot_luas.div(total_luas, axis=0).fillna(0)

# Gabungkan & beri prefix agar dapat dibedakan
pivot_cl = pd.concat([
    pivot_log.add_prefix('log_prod_'),
    pivot_prop.add_prefix('prop_luas_'),
], axis=1)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(pivot_cl)

print(f"Feature matrix : {pivot_cl.shape}  ({pivot_cl.shape[0]} kecamatan × {pivot_cl.shape[1]} fitur)")
print(f"Kecamatan      : {pivot_cl.index.tolist()}")

In [ ]:
# ── 3.2 Penentuan Jumlah Cluster Optimal ─────────────────────
inertias, silhouettes, db_scores = [], [], []
K_range = range(2, 10)

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbl))
    db_scores.append(davies_bouldin_score(X_scaled, lbl))

# K optimal = silhouette tertinggi
K_OPTIMAL = int(K_range[int(np.argmax(silhouettes))])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(K_range, inertias,   'bo-', linewidth=2.2, markersize=9)
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].axvline(x=K_OPTIMAL, color='red', linestyle='--', alpha=0.6, label=f'k={K_OPTIMAL} (dipilih)')
axes[0].legend()

axes[1].plot(K_range, silhouettes,'gs-', linewidth=2.2, markersize=9)
axes[1].set_title('Silhouette Score (↑ lebih baik)')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].axvline(x=K_OPTIMAL, color='red', linestyle='--', alpha=0.6)

axes[2].plot(K_range, db_scores,  'r^-', linewidth=2.2, markersize=9)
axes[2].set_title('Davies-Bouldin Score (↓ lebih baik)')
axes[2].set_xlabel('k'); axes[2].set_ylabel('DB Score')
axes[2].axvline(x=K_OPTIMAL, color='red', linestyle='--', alpha=0.6)

plt.suptitle('Evaluasi Jumlah Cluster Optimal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'cluster_01_elbow_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRingkasan Metrik per k:")
print(f"{'k':>3} | {'Silhouette':>12} | {'Davies-Bouldin':>15}")
print("-" * 36)
for k, sil, db in zip(K_range, silhouettes, db_scores):
    mark = " ◄ DIPILIH" if k == K_OPTIMAL else ""
    print(f"{k:>3} | {sil:>12.4f} | {db:>15.4f}{mark}")
print(f"\nK optimal = {K_OPTIMAL} (silhouette tertinggi: {max(silhouettes):.4f})")

In [ ]:
# ── 3.3 K-Means Final ────────────────────────────────────────
km_final   = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
labels_km  = km_final.fit_predict(X_scaled)

sil_km = silhouette_score(X_scaled, labels_km)
db_km  = davies_bouldin_score(X_scaled, labels_km)

print(f"K-Means — k = {K_OPTIMAL}")
print(f"  Silhouette Score : {sil_km:.4f}")
print(f"  Davies-Bouldin   : {db_km:.4f}")
print()

# Distribusi kecamatan per cluster
result_km = pd.DataFrame({'kecamatan': pivot_cl.index, 'cluster_id': labels_km})
for c in sorted(result_km['cluster_id'].unique()):
    kecs = sorted(result_km[result_km['cluster_id'] == c]['kecamatan'].tolist())
    print(f"  Cluster {c} ({len(kecs)} kec): {kecs}")

In [ ]:
# ── 3.4 Auto-labeling dari Centroid (rata-rata total produksi) ──
# Kembalikan centroid log ke skala asli: expm1(nilai log)
tanaman_cols = df['tanaman'].unique()
centroid_raw = pd.DataFrame(
    scaler.inverse_transform(km_final.cluster_centers_),
    columns=pivot_cl.columns
)
# Ambil hanya kolom log_prod_ untuk interpretasi skala produksi
log_cols = [c for c in centroid_raw.columns if c.startswith('log_prod_')]
centroid_prod = centroid_raw[log_cols].copy()
centroid_prod.columns = [c.replace('log_prod_','') for c in log_cols]
centroid_prod = np.expm1(centroid_prod).clip(lower=0).round(2)
centroid_prod.index.name = 'cluster'

print("Centroid — Estimasi rata-rata produksi (ton/tahun) per tanaman:")
print(centroid_prod)
print()

# Total produksi rata-rata per cluster → urutan untuk label
centroid_prod['_total'] = centroid_prod.sum(axis=1)
rank_order = centroid_prod['_total'].rank(method='first').astype(int)
n_cl = K_OPTIMAL
label_map_base = {1:'Potensi Rendah', 2:'Potensi Sedang', 3:'Potensi Tinggi', 4:'Potensi Sangat Tinggi'}

# Fallback jika K > 4
def auto_label(rank, n):
    if n <= 4:
        return label_map_base.get(rank, f'Cluster (rank {rank})')
    pct = rank / n
    if pct <= 0.25: return 'Potensi Rendah'
    if pct <= 0.50: return 'Potensi Sedang'
    if pct <= 0.75: return 'Potensi Tinggi'
    return 'Potensi Sangat Tinggi'

CLUSTER_LABELS = {c: auto_label(rank_order[c], n_cl) for c in range(n_cl)}
centroid_prod.drop(columns='_total', inplace=True)

print("Label cluster otomatis (berdasarkan total produksi centroid):")
for c, lbl in CLUSTER_LABELS.items():
    print(f"  Cluster {c}: {lbl}  (total centroid: {centroid_prod.loc[c].sum():.1f} ton)")
print()
print("⚠️  Verifikasi label ini dengan konteks lokal sebelum deploy!")

In [ ]:
# ── 3.5 Agglomerative + Dendrogram ───────────────────────────
agg        = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)
sil_agg    = silhouette_score(X_scaled, labels_agg)
db_agg     = davies_bouldin_score(X_scaled, labels_agg)

print(f"Agglomerative — k = {K_OPTIMAL}")
print(f"  Silhouette : {sil_agg:.4f}")
print(f"  DB Score   : {db_agg:.4f}")

linked = linkage(X_scaled, method='ward')
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(linked, labels=pivot_cl.index.tolist(), orientation='top',
           distance_sort='descending', show_leaf_counts=True, ax=ax,
           color_threshold=linked[-K_OPTIMAL+1, 2], leaf_rotation=45)
ax.set_title(f'Dendrogram — Agglomerative Clustering (Ward)\nk={K_OPTIMAL}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Kecamatan'); ax.set_ylabel('Euclidean Distance')
ax.axhline(y=linked[-K_OPTIMAL+1, 2], color='red', linestyle='--', alpha=0.5,
           label=f'Cut-off k={K_OPTIMAL}')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'cluster_02_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ cluster_02_dendrogram.png")

In [ ]:
# ── 3.6 Perbandingan K-Means vs Agglomerative ────────────────
comparison_cl = pd.DataFrame({
    'Metode'            : ['K-Means', 'Agglomerative (Ward)'],
    'Silhouette (↑)'    : [round(sil_km, 4), round(sil_agg, 4)],
    'Davies-Bouldin (↓)': [round(db_km,  4), round(db_agg,  4)],
    'Keterangan'        : ['Model utama (deploy)', 'Pembanding / validasi'],
})
print(comparison_cl.to_string(index=False))
print()
if sil_km >= sil_agg:
    print("✅ K-Means dipilih sebagai model utama")
else:
    print("ℹ️  Agglomerative punya Silhouette lebih tinggi — pertimbangkan sebagai model utama")

In [ ]:
# ── 3.7 Visualisasi 2D (PCA) ─────────────────────────────────
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
kec_names    = pivot_cl.index.tolist()
COLORS_CLUST = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22','#34495e']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (labels, title) in zip(axes, [
    (labels_km,  f'K-Means (k={K_OPTIMAL})'),
    (labels_agg, f'Agglomerative (k={K_OPTIMAL})'),
]):
    for c in sorted(np.unique(labels)):
        mask = labels == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=COLORS_CLUST[c], label=f'Cluster {c} — {CLUSTER_LABELS[c]}',
                   s=130, edgecolors='white', linewidth=0.8, alpha=0.92, zorder=3)
        for i, name in enumerate(kec_names):
            if mask[i]:
                ax.annotate(name, (X_pca[i, 0], X_pca[i, 1]),
                            textcoords='offset points', xytext=(5, 4),
                            fontsize=7, color='#333333')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(title='Cluster', framealpha=0.85, fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Visualisasi 2D Clustering Kecamatan — PCA Projection',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'cluster_03_pca_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ cluster_03_pca_2d.png")

In [ ]:
# ── 3.8 Simpan Hasil Clustering ──────────────────────────────
# Rata-rata produksi aktual per kecamatan (bukan centroid)
prod_mean_kec = df.groupby('kecamatan')['produksi_ton'].sum() / df['Tahun'].nunique()

result_cl = pd.DataFrame({
    'kecamatan'           : pivot_cl.index,
    'cluster_id'          : labels_km,
    'cluster_agg_id'      : labels_agg,
    'label_cluster'       : [CLUSTER_LABELS[c] for c in labels_km],
    'label_cluster_agg'   : [CLUSTER_LABELS[c] for c in labels_agg],
    'total_produksi_mean' : [round(prod_mean_kec.get(k, 0), 2) for k in pivot_cl.index],
})

print("Hasil Clustering Kecamatan:")
print(result_cl[['kecamatan','cluster_id','label_cluster','total_produksi_mean']].to_string(index=False))

result_cl.to_csv(OUTPUT_DIR + 'output_clustering_pangan.csv', index=False)
print(f"\n✅ Tersimpan: output_clustering_pangan.csv")

## 4. Time-Series Forecast

**Strategi**:
| Split | Tahun | Peran |
|-------|-------|-------|
| Train | 2020–2023 | Latih model |
| Validasi | 2024–2025 | Rolling-origin CV (2 fold) |
| Forecast | 2026–2027 | Prediksi final (seluruh data) |

**Model kandidat**: Naive (baseline), Mean-3, Drift, Linear, Holt ETS

**Pemilihan model**: WAPE terkecil di level **tanaman** (bukan per series) → menghindari overfit pada 1-2 titik uji.

**Output**: tabel long `aktual + forecast` dengan kolom `lower/upper` (interval empiris 80%).

In [ ]:
# ── 4.1 Konfigurasi & Fungsi Model Kandidat ─────────────────
TRAIN_UPTO   = 2023   # tahun terakhir fold pertama
HORIZON      = 2      # forecast 2026–2027
FORECAST_YRS = [2026, 2027]
MIN_TRAIN    = 3      # minimal titik untuk melatih model selain naive

TARGET_COLS = ['luas_panen_ha', 'produktivitas_kuintal_ha']
# Produksi = luas * produktivitas / 10 → diturunkan, bukan diprediksi langsung

# ── Kandidat model ────────────────────────────────────────────
def _naive(y, yr, h):
    return np.full(h, float(y[-1]))

def _mean3(y, yr, h):
    return np.full(h, float(np.mean(y[-min(3,len(y)):])))

def _drift(y, yr, h):
    slope = (y[-1] - y[0]) / max(len(y) - 1, 1)
    return np.clip(y[-1] + slope * np.arange(1, h+1), 0, None)

def _linear(y, yr, h):
    x = np.array(yr, float)
    slope, intercept, *_ = linregress(x, y)
    return np.clip(intercept + slope * np.arange(x[-1]+1, x[-1]+1+h), 0, None)

def _ets(y, yr, h):
    if len(y) < 3 or y.sum() == 0:
        raise ValueError("series terlalu pendek untuk ETS")
    m = ExponentialSmoothing(y, trend='add', damped_trend=True,
                             initialization_method='estimated').fit(optimized=True)
    return np.clip(m.forecast(h), 0, None)

CANDIDATES  = ['naive','mean3','drift','linear','ets']
MIN_POINTS  = {'naive':1,'mean3':1,'drift':2,'linear':3,'ets':3}
MODEL_FUNCS = {'naive':_naive,'mean3':_mean3,'drift':_drift,'linear':_linear,'ets':_ets}

def run_model(name, y, yr, h):
    """Jalankan satu model, return (pred_array, catatan|None)."""
    y  = np.asarray(y, float)
    yr = np.asarray(yr, float)
    if len(y) == 0:
        return np.zeros(h), 'series_kosong→nol'
    if len(y) < MIN_POINTS[name]:
        return np.full(h, max(float(y[-1]), 0.0)), f'data_kurang({len(y)})→naive'
    try:
        pred = np.asarray(MODEL_FUNCS[name](y, yr, h), float)
        if not np.all(np.isfinite(pred)):
            raise FloatingPointError('non-finite')
        return np.clip(pred, 0, None), None
    except Exception as e:
        return np.full(h, max(float(y[-1]), 0.0)), f'{name}_error:{type(e).__name__}→naive'

print("✅ Fungsi model kandidat siap")
print(f"   Model : {CANDIDATES}")
print(f"   Target: {TARGET_COLS}")

In [ ]:
# ── 4.2 Rolling-Origin Cross-Validation (2 fold) ─────────────
# Fold 1: train 2020-2023, test 2024-2025
# Fold 2: train 2020-2024, test 2025
years_all = sorted(df['Tahun'].unique())
origins   = [y for y in years_all[:-1] if sum(yr <= y for yr in years_all) >= MIN_TRAIN]

cv_rows  = []
cv_log   = []   # catat semua fallback / error

for target in TARGET_COLS:
    for kec in kecamatan_list:
        for tan in tanaman_list:
            g   = df[(df['kecamatan']==kec) & (df['tanaman']==tan)].sort_values('Tahun')
            yrs = g['Tahun'].values
            if target == 'produktivitas_kuintal_ha':
                g_valid = g[g['luas_panen_ha'] > 0]
            else:
                g_valid = g
            for origin_yr in origins:
                tr = g_valid[g_valid['Tahun'] <= origin_yr]
                if len(tr) < MIN_TRAIN:
                    continue
                ty  = tr['Tahun'].values
                tv  = tr[target].values.astype(float)
                h   = HORIZON
                for model in CANDIDATES:
                    pred, note = run_model(model, tv, ty, h)
                    if note:
                        cv_log.append(dict(kecamatan=kec, tanaman=tan, target=target,
                                           origin=origin_yr, model=model, note=note))
                    for step in range(1, h+1):
                        actual_yr = origin_yr + step
                        row_act   = g_valid[g_valid['Tahun'] == actual_yr]
                        if row_act.empty:
                            continue
                        actual_val = float(row_act[target].values[0])
                        luas_val   = float(row_act['luas_panen_ha'].values[0]) if 'luas_panen_ha' in row_act else 1.0
                        cv_rows.append(dict(
                            origin=origin_yr, h=step, tahun=actual_yr,
                            kecamatan=kec, tanaman=tan, target=target,
                            model=model, actual=actual_val, pred=float(pred[step-1]),
                            luas_actual=luas_val
                        ))

cv_df = pd.DataFrame(cv_rows)
print(f"Total baris CV   : {len(cv_df):,}")
print(f"Total fallback   : {len(cv_log)}")
if cv_log:
    log_df = pd.DataFrame(cv_log)
    print("\nFrekuensi fallback per model:")
    print(log_df.groupby('model')['note'].count().sort_values(ascending=False).to_string())

In [ ]:
# ── 4.3 Seleksi Model per Tanaman (WAPE Pooled) ──────────────
# WAPE = Weighted Absolute Percentage Error (lebih stabil dari MAPE saat aktual ≈ 0)
# Bobot = luas_panen untuk produktivitas, 1 untuk luas
# Model dipilih di level TANAMAN (bukan per series) untuk menghindari overfit

def compute_wape(sub_df):
    """WAPE dari sub-DataFrame cv_df."""
    w   = np.where(sub_df['target'] == 'produktivitas_kuintal_ha',
                   sub_df['luas_actual'], 1.0)
    err = w * (sub_df['actual'] - sub_df['pred']).abs()
    act = w *  sub_df['actual'].abs()
    return float(err.sum() / act.sum()) if act.sum() > 0 else np.nan

MARGIN = 0.02   # model non-naive harus lebih baik >= 2% untuk dipilih

selected_rows = []
wape_all_rows = []

for target in TARGET_COLS:
    for tan in tanaman_list:
        sub = cv_df[(cv_df['tanaman']==tan) & (cv_df['target']==target)]
        naive_wape = compute_wape(sub[sub['model']=='naive'])
        best_model, best_wape = 'naive', naive_wape

        for model in CANDIDATES:
            w = compute_wape(sub[sub['model']==model])
            wape_all_rows.append(dict(tanaman=tan, target=target, model=model, wape=w))
            if model == 'naive':
                continue
            if np.isfinite(w) and np.isfinite(naive_wape) and w <= naive_wape * (1 - MARGIN):
                if np.isnan(best_wape) or w < best_wape:
                    best_model, best_wape = model, w

        improve = (1 - best_wape/naive_wape) if np.isfinite(naive_wape) and naive_wape>0 else np.nan
        selected_rows.append(dict(tanaman=tan, target=target, model=best_model,
                                  wape=best_wape, wape_naive=naive_wape,
                                  perbaikan_vs_naive=round(improve, 4) if np.isfinite(improve) else None))

selected_df  = pd.DataFrame(selected_rows)
wape_all_df  = pd.DataFrame(wape_all_rows)

print("Model Terpilih per (Tanaman × Target):")
print(selected_df[['tanaman','target','model','wape','wape_naive','perbaikan_vs_naive']].to_string(index=False))

In [ ]:
# ── 4.4 Visualisasi WAPE per Model ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_colors = {'naive':'#95a5a6','mean3':'#3498db','drift':'#e74c3c',
                'linear':'#2ecc71','ets':'#f39c12'}

for ax, target in zip(axes, TARGET_COLS):
    sub = wape_all_df[wape_all_df['target']==target]
    wape_means = sub.groupby('model')['wape'].mean() * 100

    bars = ax.bar(wape_means.index,
                  wape_means.values,
                  color=[model_colors.get(m,'#777') for m in wape_means.index],
                  edgecolor='white', width=0.5)
    ax.set_title(f'Rata-rata WAPE — {target.replace("_"," ").title()}')
    ax.set_ylabel('WAPE (%)')
    ax.set_xlabel('Model')
    for bar, v in zip(bars, wape_means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)
    ax.axhline(y=15, color='orange', linestyle='--', alpha=0.6, label='Ambang 15%')
    ax.legend(fontsize=9)

plt.suptitle('Perbandingan Akurasi Model — WAPE (Rolling-Origin CV)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'ts_01_model_comparison_wape.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ ts_01_model_comparison_wape.png")

In [ ]:
# ── 4.5 Interval Prediksi Empiris (80%) ──────────────────────
# Rasio aktual/prediksi dari CV → kuantil 10% dan 90%
# Pooling per tanaman; fallback ke semua tanaman jika n < 20

INTERVAL_LEVEL = 0.80
LO_Q = (1 - INTERVAL_LEVEL) / 2       # 0.10
HI_Q = 1 - LO_Q                       # 0.90
MIN_N_RATIO = 20

# Hitung prediksi produksi di CV: luas × produktivitas / 10
parts = {}
for tgt in TARGET_COLS:
    sub = cv_df[cv_df['target']==tgt].copy()
    sub = sub.merge(selected_df[selected_df['target']==tgt][['tanaman','model']],
                    on=['tanaman','model'])
    parts[tgt] = sub[['origin','h','tahun','kecamatan','tanaman','pred']]                   .rename(columns={'pred': f'pred_{tgt}'})

cv_prod = parts['luas_panen_ha'].merge(
            parts['produktivitas_kuintal_ha'],
            on=['origin','h','tahun','kecamatan','tanaman'])
cv_prod['pred_produksi'] = cv_prod['pred_luas_panen_ha'] * cv_prod['pred_produktivitas_kuintal_ha'] / 10

act_prod = df[['Tahun','kecamatan','tanaman','produksi_ton']]             .rename(columns={'Tahun':'tahun','produksi_ton':'actual_produksi'})
cv_prod  = cv_prod.merge(act_prod, on=['tahun','kecamatan','tanaman'])

ok       = cv_prod[(cv_prod['pred_produksi'] > 0) & (cv_prod['actual_produksi'] > 0)].copy()
ok['ratio'] = ok['actual_produksi'] / ok['pred_produksi']

interval_rows = []
for tan in tanaman_list:
    for h in range(1, HORIZON+1):
        r = ok[(ok['tanaman']==tan) & (ok['h']==h)]['ratio'].values
        src = 'tanaman'
        if len(r) < MIN_N_RATIO:
            r, src = ok[ok['h']==h]['ratio'].values, 'pooled'
        if len(r) < MIN_N_RATIO:
            lo, hi, src = 0.70, 1.30, 'default'
        else:
            lo, hi = float(np.quantile(r, LO_Q)), float(np.quantile(r, HI_Q))
        interval_rows.append(dict(tanaman=tan, h=h,
                                  ratio_lo=min(lo, 1.0), ratio_hi=max(hi, 1.0),
                                  n=len(r), sumber=src, level=INTERVAL_LEVEL))

interval_df = pd.DataFrame(interval_rows)
print("Interval rasio empiris 80% per tanaman x horizon:")
print(interval_df[['tanaman','h','ratio_lo','ratio_hi','n','sumber']].to_string(index=False))

In [ ]:
# ── 4.6 Forecast Final 2026–2027 ─────────────────────────────
sel_map   = {(r.tanaman, r.target): r.model for r in selected_df.itertuples()}
iv_map    = {(r.tanaman, r.h): (r.ratio_lo, r.ratio_hi) for r in interval_df.itertuples()}
last_year = int(df['Tahun'].max())

final_rows = []
fallback_final = []

for (kec, tan), g in df.groupby(['kecamatan','tanaman']):
    g = g.sort_values('Tahun')
    preds = {}
    for tgt in TARGET_COLS:
        gv     = g[g['luas_panen_ha'] > 0] if tgt == 'produktivitas_kuintal_ha' else g
        ty, tv = gv['Tahun'].values, gv[tgt].values.astype(float)
        pred, note = run_model(sel_map[(tan, tgt)], tv, ty, HORIZON)
        preds[tgt] = pred
        if note:
            fallback_final.append(dict(kecamatan=kec, tanaman=tan, target=tgt, note=note))

    luas_pred = preds['luas_panen_ha']
    prod_pred  = preds['produktivitas_kuintal_ha']
    for h in range(1, HORIZON+1):
        yr           = last_year + h
        produksi_fc  = luas_pred[h-1] * prod_pred[h-1] / 10
        lo_r, hi_r   = iv_map[(tan, h)]
        final_rows.append(dict(tahun=yr, kecamatan=kec, tanaman=tan,
                               luas_panen_ha=round(luas_pred[h-1], 3),
                               produktivitas_kuintal_ha=round(prod_pred[h-1], 3),
                               produksi_ton=round(produksi_fc, 3),
                               produksi_lower=round(produksi_fc * lo_r, 3),
                               produksi_upper=round(produksi_fc * hi_r, 3),
                               model_luas=sel_map[(tan,'luas_panen_ha')],
                               model_prod_v=sel_map[(tan,'produktivitas_kuintal_ha')],
                               tipe='forecast'))

forecast_df = pd.DataFrame(final_rows)

print(f"Total baris forecast : {len(forecast_df):,}")
print(f"Fallback final       : {len(fallback_final)}")
print()
print(forecast_df.head(8).to_string(index=False))

In [ ]:
# ── 4.7 Gabungkan Aktual + Forecast ──────────────────────────
hist_df = df[['Tahun','kecamatan','tanaman',
              'luas_panen_ha','produktivitas_kuintal_ha','produksi_ton']].copy()
hist_df.rename(columns={'Tahun':'tahun'}, inplace=True)
hist_df['produksi_lower'] = np.nan
hist_df['produksi_upper'] = np.nan
hist_df['model_luas']     = 'aktual'
hist_df['model_prod_v']   = 'aktual'
hist_df['tipe']           = 'aktual'

final_ts_df = pd.concat([hist_df, forecast_df], ignore_index=True)
final_ts_df = final_ts_df.sort_values(['kecamatan','tanaman','tahun']).reset_index(drop=True)

print(f"Total baris output   : {len(final_ts_df):,}")
print(f"Tipe data            : {final_ts_df['tipe'].value_counts().to_dict()}")

# Export CSV
final_ts_df.to_csv(OUTPUT_DIR + 'output_timeseries_pangan.csv', index=False)
selected_df.to_csv(OUTPUT_DIR + 'output_model_comparison.csv', index=False)
print("\n✅ output_timeseries_pangan.csv")
print("✅ output_model_comparison.csv")

In [ ]:
# ── 4.8 Visualisasi Contoh Forecast ──────────────────────────
kec_samples  = ['Pare', 'Gurah', 'Kepung', 'Ngadiluwih']
# Fallback jika kecamatan tidak ada di data
kec_samples  = [k for k in kec_samples if k in kecamatan_list][:4]
if len(kec_samples) < 4:
    kec_samples = kecamatan_list[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, kec in zip(axes, kec_samples):
    sample  = final_ts_df[(final_ts_df['kecamatan']==kec) & (final_ts_df['tanaman']=='Padi')]
    aktual  = sample[sample['tipe']=='aktual']
    fc      = sample[sample['tipe']=='forecast']

    ax.plot(aktual['tahun'], aktual['produksi_ton'], 'o-',
            color='#2980b9', label='Aktual', linewidth=2.2, markersize=8)
    if not fc.empty:
        # Sambungkan garis dari titik terakhir aktual
        x_conn = [aktual['tahun'].iloc[-1]] + fc['tahun'].tolist()
        y_conn = [aktual['produksi_ton'].iloc[-1]] + fc['produksi_ton'].tolist()
        ax.plot(x_conn, y_conn, 's--', color='#e74c3c',
                label='Forecast', linewidth=2.2, markersize=8)
        # Area interval
        x_iv  = fc['tahun'].values
        ax.fill_between(x_iv, fc['produksi_lower'], fc['produksi_upper'],
                        alpha=0.18, color='#e74c3c', label='Interval 80%')

    ax.axvspan(last_year + 0.5, last_year + HORIZON + 0.5, alpha=0.07, color='orange')
    ax.set_title(f'Padi — {kec}', fontweight='bold')
    ax.set_xlabel('Tahun'); ax.set_ylabel('Produksi (Ton)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.suptitle(f'Prediksi Produksi Padi 2026–2027 (interval 80%)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'ts_02_sample_forecast_padi.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ ts_02_sample_forecast_padi.png")

## 5. Rekomendasi Tanaman

**Perbaikan dari versi sebelumnya**:
- Growth rate dihitung dari **slope regresi OLS** (bukan first/last value) → tahan outlier
- Normalisasi dilakukan **per tanaman** (bukan global) → menghilangkan bias skala kedelai vs padi
- Skor disebut **"Skor Potensi Historis"** (bukan Bayesian) agar tidak menyesatkan
- Bobot default: Produksi 40% + Produktivitas 40% + Trend 20%

In [ ]:
# ── 5.1 Hitung Fitur Agregat per (Kecamatan × Tanaman) ───────
agg_rows = []
for (kec, tan), g in df.groupby(['kecamatan','tanaman']):
    g = g.sort_values('Tahun')
    yrs  = g['Tahun'].values.astype(float)
    prod = g['produksi_ton'].values.astype(float)
    prod_v = g['produktivitas_kuintal_ha'].values.astype(float)
    # Growth rate = slope OLS pada produksi (ton/tahun), dinormalisasi oleh rata-rata
    if len(yrs) >= 2 and g['produksi_ton'].mean() > 0:
        slope = linregress(yrs, prod).slope
        # Normalkan slope dengan rata-rata → growth rate relatif (%/tahun)
        growth = slope / (g['produksi_ton'].mean() + 1e-6)
    else:
        growth = 0.0

    agg_rows.append(dict(
        kecamatan         = kec,
        tanaman           = tan,
        prod_mean         = float(g['produksi_ton'].mean()),
        produktivitas_mean= float(g['produktivitas_kuintal_ha'].mean()),
        luas_mean         = float(g['luas_panen_ha'].mean()),
        growth_rate       = float(np.clip(growth, -1.0, 2.0)),
        n_tahun           = int(len(g)),
    ))

agg_rek = pd.DataFrame(agg_rows)
agg_rek_valid = agg_rek[agg_rek['prod_mean'] > 0].copy()
print(f"Kombinasi kec×tanaman dengan produksi > 0: {len(agg_rek_valid)} dari {len(agg_rek)}")
print()
print(agg_rek_valid[['kecamatan','tanaman','prod_mean','produktivitas_mean','growth_rate']]
      .sort_values('prod_mean', ascending=False).head(10).round(3).to_string(index=False))

In [ ]:
# ── 5.2 Normalisasi PER TANAMAN (menghindari bias skala) ──────
score_cols = ['prod_mean', 'produktivitas_mean', 'growth_rate']

norm_parts = []
for tan in agg_rek_valid['tanaman'].unique():
    sub = agg_rek_valid[agg_rek_valid['tanaman'] == tan].copy()
    for col in score_cols:
        mn, mx = sub[col].min(), sub[col].max()
        if mx > mn:
            sub[f'norm_{col}'] = (sub[col] - mn) / (mx - mn)
        else:
            sub[f'norm_{col}'] = 0.5   # semua kecamatan sama → skor netral
    norm_parts.append(sub)

agg_rek_valid = pd.concat(norm_parts, ignore_index=True)

# ── Bobot skor potensi historis ───────────────────────────────
W_PROD   = 0.40   # produksi rata-rata
W_PROD_V = 0.40   # produktivitas rata-rata
W_GROW   = 0.20   # tren pertumbuhan

agg_rek_valid['skor_potensi'] = (
    W_PROD   * agg_rek_valid['norm_prod_mean'] +
    W_PROD_V * agg_rek_valid['norm_produktivitas_mean'] +
    W_GROW   * agg_rek_valid['norm_growth_rate']
).round(4)

print("Contoh 10 kombinasi skor tertinggi:")
print(agg_rek_valid[['kecamatan','tanaman','prod_mean','produktivitas_mean',
                      'growth_rate','skor_potensi']]
      .sort_values('skor_potensi', ascending=False).head(10).round(3).to_string(index=False))

In [ ]:
# ── 5.3 Top-3 Rekomendasi per Kecamatan ─────────────────────
TOP_N = 3

rekomendasi = (
    agg_rek_valid
    .sort_values(['kecamatan','skor_potensi'], ascending=[True, False])
    .groupby('kecamatan').head(TOP_N)
    .reset_index(drop=True)
)
rekomendasi['rank'] = rekomendasi.groupby('kecamatan').cumcount() + 1

print(f"TOP {TOP_N} REKOMENDASI TANAMAN PER KECAMATAN")
print("=" * 65)
for kec in sorted(rekomendasi['kecamatan'].unique()):
    sub   = rekomendasi[rekomendasi['kecamatan'] == kec]
    items = [f"{row['tanaman']} ({row['skor_potensi']:.3f})" for _, row in sub.iterrows()]
    print(f"  {kec:<18}: {' | '.join(items)}")

In [ ]:
# ── 5.4 Heatmap Skor Potensi ─────────────────────────────────
pivot_skor = agg_rek_valid.pivot(
    index='kecamatan', columns='tanaman', values='skor_potensi'
).fillna(0)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(pivot_skor, annot=True, fmt='.3f', cmap='Greens',
            linewidths=0.4, linecolor='white', ax=ax,
            cbar_kws={'label': 'Skor Potensi Historis (0–1)', 'shrink': 0.8},
            vmin=0, vmax=1)
ax.set_title(
    f'Heatmap Skor Potensi Historis per Kecamatan\n'
    f'(Produksi {int(W_PROD*100)}% + Produktivitas {int(W_PROD_V*100)}% + Trend {int(W_GROW*100)}%)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Jenis Tanaman'); ax.set_ylabel('Kecamatan')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'rek_01_heatmap_skor.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ rek_01_heatmap_skor.png")

In [ ]:
# ── 5.5 Distribusi Tanaman Unggulan ──────────────────────────
rank_counts = rekomendasi.groupby(['tanaman','rank']).size().unstack(fill_value=0)
colors_bar  = ['#2ecc71','#3498db','#f39c12']

fig, ax = plt.subplots(figsize=(10, 5))
rank_counts.plot(kind='bar', ax=ax, color=colors_bar[:len(rank_counts.columns)],
                 edgecolor='white', width=0.65)
ax.set_title(f'Frekuensi Tanaman sebagai Top-{TOP_N} Rekomendasi',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Jenis Tanaman')
ax.set_ylabel('Jumlah Kecamatan')
ax.legend(title='Rank', labels=[f'Rank {r}' for r in rank_counts.columns], framealpha=0.85)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'rek_02_distribusi_unggulan.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ rek_02_distribusi_unggulan.png")

In [ ]:
# ── 5.6 Export CSV Rekomendasi ───────────────────────────────
agg_rek_valid[['kecamatan','tanaman','prod_mean','produktivitas_mean',
               'growth_rate','luas_mean','skor_potensi']].round(4)    .to_csv(OUTPUT_DIR + 'output_rekomendasi_all_scores.csv', index=False)

rekomendasi[['kecamatan','tanaman','rank','skor_potensi',
             'prod_mean','produktivitas_mean','growth_rate']].round(4)    .to_csv(OUTPUT_DIR + 'output_rekomendasi_pangan.csv', index=False)

print("✅ output_rekomendasi_all_scores.csv")
print("✅ output_rekomendasi_pangan.csv")

## 6. Export Deploy Bundle (.pkl + CSV)

Bundle `agridash_model_bundle.pkl` berisi **satu dictionary** dengan semua artefak yang
dibutuhkan API backend.

```
bundle = {
  "meta"               : {...}          # versi, tanggal, hash data, konfigurasi
  "scaler"             : StandardScaler # dipakai clustering inferensi
  "kmeans"             : KMeans         # model clustering utama
  "cluster_labels"     : {0:..., 1:...} # pemetaan id → label
  "cluster_assignments": DataFrame      # kecamatan → cluster (siap query)
  "selected_models"    : DataFrame      # model terpilih per tanaman × target
  "interval_df"        : DataFrame      # rasio interval empiris
  "forecast_long"      : DataFrame      # aktual + forecast (tabel utama dashboard)
  "forecast_kabupaten" : DataFrame      # agregasi level kabupaten
  "recommendations"    : DataFrame      # top-3 per kecamatan
  "rec_scores"         : DataFrame      # semua skor rekomendasi
  "cv_wape"            : DataFrame      # ringkasan akurasi CV
  "cv_log"             : list[dict]     # log fallback saat training
}
```

> **Cara load di API**:
> ```python
> import pickle
> with open('agridash_model_bundle.pkl', 'rb') as f:
>     bundle = pickle.load(f)
> fc = bundle['forecast_long']
> ```

In [ ]:
# ── 6.1 Agregasi Kabupaten ───────────────────────────────────
# Luas & Produksi = sum; Produktivitas = rata-rata berbobot luas
kab_rows = []
for (tan, thn, tipe), g in final_ts_df.groupby(['tanaman','tahun','tipe']):
    luas = float(g['luas_panen_ha'].sum())
    prod = float(g['produksi_ton'].sum())
    produktivitas = (prod * 10 / luas) if luas > 0 else 0.0
    prod_lo = float(g['produksi_lower'].sum()) if tipe == 'forecast' else np.nan
    prod_hi = float(g['produksi_upper'].sum()) if tipe == 'forecast' else np.nan
    kab_rows.append(dict(tanaman=tan, tahun=thn, tipe=tipe,
                         luas_panen_ha=round(luas, 3),
                         produksi_ton=round(prod, 3),
                         produktivitas_kuintal_ha=round(produktivitas, 3),
                         produksi_lower=round(prod_lo, 3) if np.isfinite(prod_lo) else np.nan,
                         produksi_upper=round(prod_hi, 3) if np.isfinite(prod_hi) else np.nan))

forecast_kab = pd.DataFrame(kab_rows).sort_values(['tanaman','tahun']).reset_index(drop=True)
print("Agregasi Kabupaten (5 baris pertama forecast):")
print(forecast_kab[forecast_kab['tipe']=='forecast'].head().to_string(index=False))

### 6b. EWS Kecamatan, Dashboard Summary & Visualisasi per Tanaman

In [ ]:
# ── 6b.1 EWS Kecamatan & Dashboard Summary per Tanaman ────────
# Menghasilkan dua tabel yang langsung dipakai UI:
#
#  ews_kecamatan   : status Normal/Waspada/Siaga per kec × tan × tahun_fc
#  dashboard_summary: satu baris per tanaman × tahun_fc untuk kartu header UI
#
# Ambang EWS (indikatif, dapat disesuaikan):
STABLE_BAND   = 0.05   # ±5%  → Stabil
DROP_WASPADA  = 0.15   # -15% → Waspada
DROP_SIAGA    = 0.30   # -30% → Siaga
MIN_BASE_TON  = 1.0    # basis < 1 ton → tidak dinilai (hindari % menyesatkan)

last_year = int(df['Tahun'].max())

# ── Hitung volatilitas YoY historis per kecamatan × tanaman ──
vol_map = {}
for (kec, tan), g in df.groupby(['kecamatan','tanaman']):
    vals = g.sort_values('Tahun')['produksi_ton'].values.astype(float)
    if len(vals) >= 2:
        yoy = np.diff(vals) / np.where(vals[:-1] > 0, vals[:-1], np.nan)
        yoy = yoy[np.isfinite(yoy)]
        vol_map[(kec, tan)] = float(np.std(yoy)) if len(yoy) > 0 else 0.0
    else:
        vol_map[(kec, tan)] = 0.0

# ── EWS per kecamatan ─────────────────────────────────────────
ews_rows = []
fc_only = final_ts_df[final_ts_df['tipe'] == 'forecast'].copy()
aktual_last = (
    final_ts_df[final_ts_df['tipe'] == 'aktual']
    .sort_values('tahun')
    .groupby(['kecamatan','tanaman'])
    .last()
    .reset_index()[['kecamatan','tanaman','produksi_ton']]
    .rename(columns={'produksi_ton':'aktual_basis'})
)

fc_with_base = fc_only.merge(aktual_last, on=['kecamatan','tanaman'])

for _, row in fc_with_base.iterrows():
    base = row['aktual_basis']
    pred = row['produksi_ton']
    vol  = vol_map.get((row['kecamatan'], row['tanaman']), 0.0)

    if base < MIN_BASE_TON:
        delta, status, risk = None, 'Basis kecil', 'Tidak dinilai'
        dalam_interval = None
    else:
        delta = (pred - base) / base
        # Status tren (dengan dead-band ±STABLE_BAND)
        if   delta >  STABLE_BAND: status = 'Meningkat'
        elif delta < -STABLE_BAND: status = 'Menurun'
        else:                      status = 'Stabil'
        # Ambang adaptif: gabung ambang tetap dengan volatilitas historis
        thr_w = max(DROP_WASPADA, vol)
        thr_s = min(DROP_SIAGA,   0.80)
        if   delta <= -thr_s: risk = 'Siaga'
        elif delta <= -thr_w: risk = 'Waspada'
        else:                 risk = 'Normal'
        # Apakah perubahan masih dalam interval ketidakpastian model?
        lo, hi = row['produksi_lower'], row['produksi_upper']
        dalam_interval = bool(lo <= base <= hi) if (pd.notna(lo) and pd.notna(hi)) else None

    ews_rows.append(dict(
        tahun          = int(row['tahun']),
        kecamatan      = row['kecamatan'],
        tanaman        = row['tanaman'],
        tahun_basis    = last_year,
        aktual_basis   = round(base, 3),
        prediksi       = round(pred, 3),
        produksi_lower = round(row['produksi_lower'], 3) if pd.notna(row['produksi_lower']) else None,
        produksi_upper = round(row['produksi_upper'], 3) if pd.notna(row['produksi_upper']) else None,
        delta_pct      = round(delta, 4) if delta is not None else None,
        volatilitas    = round(vol, 4),
        status         = status,
        risk_level     = risk,
        dalam_interval = dalam_interval,
    ))

ews_kecamatan = pd.DataFrame(ews_rows)

print(f"EWS Kecamatan — {len(ews_kecamatan)} baris")
print("\nDistribusi risk_level:")
print(ews_kecamatan['risk_level'].value_counts().to_string())
print("\nDistribusi status tren:")
print(ews_kecamatan['status'].value_counts().to_string())
print()
print("Contoh 5 baris:")
print(ews_kecamatan[['tahun','kecamatan','tanaman','aktual_basis','prediksi',
                      'delta_pct','status','risk_level']].head().to_string(index=False))

In [ ]:
# ── 6b.2 Dashboard Summary per Tanaman (kartu header UI) ─────
# Satu baris per tanaman × tahun_fc → data kartu "Potensi Risiko" di dashboard
#
# Kolom yang langsung dipetakan ke komponen UI:
#   tanaman, tahun, aktual_basis_ton, prediksi_ton, delta_pct, status_tren
#   n_kec_total, n_kec_dinilai, n_kec_naik, n_kec_stabil, n_kec_turun
#   n_kec_waspada, n_kec_siaga, dalam_interval, interpretasi

summary_rows = []
for tan in sorted(ews_kecamatan['tanaman'].unique()):
    # Ambil basis dari agregasi kabupaten
    kab_aktual = forecast_kab[
        (forecast_kab['tanaman'] == tan) & (forecast_kab['tipe'] == 'aktual')
    ].sort_values('tahun')

    if kab_aktual.empty:
        continue
    basis_kab     = float(kab_aktual.iloc[-1]['produksi_ton'])
    basis_tahun   = int(kab_aktual.iloc[-1]['tahun'])

    for tahun_fc in sorted(ews_kecamatan['tahun'].unique()):
        sub_ews = ews_kecamatan[
            (ews_kecamatan['tanaman'] == tan) &
            (ews_kecamatan['tahun']   == tahun_fc)
        ]
        kab_fc = forecast_kab[
            (forecast_kab['tanaman'] == tan) &
            (forecast_kab['tahun']   == tahun_fc) &
            (forecast_kab['tipe']    == 'forecast')
        ]
        if kab_fc.empty or sub_ews.empty:
            continue

        pred_kab  = float(kab_fc.iloc[0]['produksi_ton'])
        lower_kab = float(kab_fc.iloc[0]['produksi_lower']) if pd.notna(kab_fc.iloc[0]['produksi_lower']) else None
        upper_kab = float(kab_fc.iloc[0]['produksi_upper']) if pd.notna(kab_fc.iloc[0]['produksi_upper']) else None

        delta_kab = (pred_kab - basis_kab) / basis_kab if basis_kab > 0 else None

        if delta_kab is None:
            status_kab = 'Tidak dinilai'
        elif delta_kab > STABLE_BAND:
            status_kab = 'Meningkat'
        elif delta_kab < -STABLE_BAND:
            status_kab = 'Menurun'
        else:
            status_kab = 'Stabil'

        # Hitung dalam interval (apakah basis kabupaten masuk interval prediksi)
        dalam_iv = bool(lower_kab <= basis_kab <= upper_kab) if (lower_kab and upper_kab) else None

        # Hitung jumlah kecamatan per status (hanya yang dinilai)
        dinilai = sub_ews[sub_ews['status'] != 'Basis kecil']
        n_total    = len(sub_ews)
        n_dinilai  = len(dinilai)
        n_naik     = int((dinilai['status'] == 'Meningkat').sum())
        n_stabil   = int((dinilai['status'] == 'Stabil').sum())
        n_turun    = int((dinilai['status'] == 'Menurun').sum())
        n_waspada  = int((dinilai['risk_level'] == 'Waspada').sum())
        n_siaga    = int((dinilai['risk_level'] == 'Siaga').sum())

        # Teks interpretasi otomatis (siap tampil di UI tanpa modifikasi)
        delta_str = f"{delta_kab:+.1%}" if delta_kab is not None else "N/A"
        interp = (
            f"Prediksi produksi {tan} tahun {tahun_fc}: "
            f"{pred_kab:,.2f} ton ({delta_str} vs aktual {basis_tahun} {basis_kab:,.2f} ton). "
            f"Status kabupaten: {status_kab}. "
            f"{n_turun} dari {n_dinilai} kecamatan yang dapat dinilai diprediksi menurun "
            f"(>{STABLE_BAND:.0%}). "
        )
        if n_siaga > 0:
            interp += f"{n_siaga} kecamatan berstatus Siaga (penurunan ≥{DROP_SIAGA:.0%}). "
        if n_waspada > 0:
            interp += f"{n_waspada} kecamatan berstatus Waspada (penurunan ≥{DROP_WASPADA:.0%}). "
        if dalam_iv:
            interp += "Perubahan masih berada dalam rentang ketidakpastian model (indikatif)."
        elif dalam_iv is False:
            interp += "Perubahan melewati batas interval prediksi — perlu pemantauan lebih lanjut."

        summary_rows.append(dict(
            tanaman            = tan,
            tahun              = tahun_fc,
            tahun_basis        = basis_tahun,
            aktual_basis_ton   = round(basis_kab, 3),
            prediksi_ton       = round(pred_kab, 3),
            produksi_lower     = round(lower_kab, 3) if lower_kab else None,
            produksi_upper     = round(upper_kab, 3) if upper_kab else None,
            delta_pct          = round(delta_kab, 4) if delta_kab is not None else None,
            status_tren        = status_kab,
            dalam_interval     = dalam_iv,
            n_kec_total        = n_total,
            n_kec_dinilai      = n_dinilai,
            n_kec_naik         = n_naik,
            n_kec_stabil       = n_stabil,
            n_kec_turun        = n_turun,
            n_kec_waspada      = n_waspada,
            n_kec_siaga        = n_siaga,
            interpretasi       = interp,
        ))

dashboard_summary = pd.DataFrame(summary_rows)

print(f"Dashboard Summary — {len(dashboard_summary)} baris ({dashboard_summary['tanaman'].nunique()} tanaman × {dashboard_summary['tahun'].nunique()} tahun forecast)")
print()
print(dashboard_summary[['tanaman','tahun','aktual_basis_ton','prediksi_ton',
                          'delta_pct','status_tren','n_kec_turun','n_kec_waspada','n_kec_siaga']].to_string(index=False))
print()
print("── Contoh teks interpretasi (Padi) ──")
padi_row = dashboard_summary[dashboard_summary['tanaman'].str.lower().str.contains('padi')]
if not padi_row.empty:
    print(padi_row.iloc[0]['interpretasi'])

In [ ]:
# ── 6b.3 Visualisasi Forecast Semua Tanaman (level Kabupaten) ─
# Satu panel per tanaman → semua grafik tersimpan sebagai PNG
# dan juga ditampilkan dalam satu figure grid

n_tan   = len(tanaman_list)
n_cols  = 2
n_rows  = (n_tan + 1) // n_cols
clr_act = '#2980b9'
clr_fc  = '#e74c3c'

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = axes.flatten()

for idx, tan in enumerate(tanaman_list):
    ax = axes[idx]

    kab_tan = forecast_kab[forecast_kab['tanaman'] == tan].sort_values('tahun')
    aktual  = kab_tan[kab_tan['tipe'] == 'aktual']
    fc      = kab_tan[kab_tan['tipe'] == 'forecast']

    # Garis aktual
    ax.plot(aktual['tahun'], aktual['produksi_ton'], 'o-',
            color=clr_act, linewidth=2.2, markersize=8, label='Aktual', zorder=4)

    if not fc.empty:
        # Sambung dari titik terakhir aktual
        x_conn = [aktual['tahun'].iloc[-1]] + fc['tahun'].tolist()
        y_conn = [aktual['produksi_ton'].iloc[-1]] + fc['produksi_ton'].tolist()
        ax.plot(x_conn, y_conn, 's--', color=clr_fc,
                linewidth=2.2, markersize=8, label='Forecast', zorder=4)

        # Area interval 80%
        x_iv = fc['tahun'].values
        lo   = fc['produksi_lower'].values
        hi   = fc['produksi_upper'].values
        if not (np.all(np.isnan(lo)) or np.all(np.isnan(hi))):
            ax.fill_between(x_iv, lo, hi, alpha=0.18, color=clr_fc, label='Interval 80%')

        # Anotasi nilai prediksi
        for _, r in fc.iterrows():
            ax.annotate(f"{r['produksi_ton']:,.1f}",
                        xy=(r['tahun'], r['produksi_ton']),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', fontsize=8, color=clr_fc, fontweight='bold')

    # Zona forecast
    ax.axvspan(last_year + 0.5, last_year + HORIZON + 0.5, alpha=0.07, color='orange')

    # Ambil status dari dashboard_summary untuk anotasi
    ds = dashboard_summary[dashboard_summary['tanaman'] == tan]
    if not ds.empty:
        row0 = ds.sort_values('tahun').iloc[0]
        badge_color = '#27ae60' if row0['status_tren'] == 'Meningkat' else \
                      '#e74c3c' if row0['status_tren'] == 'Menurun'   else '#f39c12'
        ax.set_title(f"{tan}  ●  {row0['status_tren']}  ({row0['delta_pct']:+.1%})",
                     fontweight='bold', color=badge_color)
    else:
        ax.set_title(tan, fontweight='bold')

    ax.set_xlabel('Tahun'); ax.set_ylabel('Produksi (Ton)')
    ax.legend(fontsize=8, framealpha=0.8)
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Sembunyikan panel kosong
for idx in range(n_tan, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Forecast Produksi 2026–2027 — Semua Tanaman Pangan\nKabupaten Kediri (interval 80%)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
out_path = OUTPUT_DIR + 'ts_03_forecast_semua_tanaman.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ ts_03_forecast_semua_tanaman.png")

# Simpan juga per-tanaman (satu file PNG per komoditas → langsung pakai di UI)
for tan in tanaman_list:
    fig_s, ax_s = plt.subplots(figsize=(10, 5))
    kab_tan = forecast_kab[forecast_kab['tanaman'] == tan].sort_values('tahun')
    aktual  = kab_tan[kab_tan['tipe'] == 'aktual']
    fc      = kab_tan[kab_tan['tipe'] == 'forecast']

    ax_s.plot(aktual['tahun'], aktual['produksi_ton'], 'o-',
              color=clr_act, linewidth=2.5, markersize=9, label='Aktual', zorder=4)
    if not fc.empty:
        x_conn = [aktual['tahun'].iloc[-1]] + fc['tahun'].tolist()
        y_conn = [aktual['produksi_ton'].iloc[-1]] + fc['produksi_ton'].tolist()
        ax_s.plot(x_conn, y_conn, 's--', color=clr_fc,
                  linewidth=2.5, markersize=9, label='Forecast', zorder=4)
        x_iv = fc['tahun'].values
        lo   = fc['produksi_lower'].values
        hi   = fc['produksi_upper'].values
        if not (np.all(np.isnan(lo)) or np.all(np.isnan(hi))):
            ax_s.fill_between(x_iv, lo, hi, alpha=0.20, color=clr_fc, label='Interval 80%')
        for _, r in fc.iterrows():
            ax_s.annotate(f"{r['produksi_ton']:,.2f} ton",
                          xy=(r['tahun'], r['produksi_ton']),
                          xytext=(0, 12), textcoords='offset points',
                          ha='center', fontsize=9.5, color=clr_fc, fontweight='bold')
            ax_s.annotate(f"[{r['produksi_lower']:,.1f} – {r['produksi_upper']:,.1f}]",
                          xy=(r['tahun'], r['produksi_ton']),
                          xytext=(0, -16), textcoords='offset points',
                          ha='center', fontsize=7.5, color='#777')

    ax_s.axvspan(last_year + 0.5, last_year + HORIZON + 0.5, alpha=0.07, color='orange')
    ds = dashboard_summary[dashboard_summary['tanaman'] == tan]
    subtitle = ''
    if not ds.empty:
        r = ds.sort_values('tahun').iloc[0]
        subtitle = f"  |  Prediksi {r['tahun']}: {r['delta_pct']:+.1%}  |  Status: {r['status_tren']}"
    ax_s.set_title(f'Forecasting Produksi — {tan}{subtitle}', fontweight='bold')
    ax_s.set_xlabel('Tahun'); ax_s.set_ylabel('Produksi (Ton)')
    ax_s.legend(fontsize=9); ax_s.grid(True, alpha=0.25)
    ax_s.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    slug = tan.lower().replace(' ','_').replace('/','_')
    fig_s.tight_layout()
    fig_s.savefig(OUTPUT_DIR + f'ts_forecast_{slug}.png', dpi=150, bbox_inches='tight')
    plt.close(fig_s)

print(f"✅ PNG per-tanaman ({len(tanaman_list)} file): ts_forecast_<nama_tanaman>.png")

In [ ]:
# ── 6.2 Ringkasan WAPE per Tanaman ───────────────────────────
wape_summary = selected_df[['tanaman','target','model','wape','wape_naive','perbaikan_vs_naive']]                 .copy().round(4)
print("Ringkasan WAPE (model terpilih):")
print(wape_summary.to_string(index=False))

In [ ]:
# ── 6.3 Buat dan Simpan Bundle PKL ───────────────────────────
import platform

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

# Hash data sumber untuk reproduksi
data_hash = hashlib.sha256(
    pd.util.hash_pandas_object(df, index=False).values.tobytes()
).hexdigest()

meta = dict(
    model_version  = '2.0.0',
    run_id         = RUN_ID,
    created_utc    = datetime.now(timezone.utc).isoformat(timespec='seconds'),
    data_sha256    = data_hash,
    tahun_data     = [int(df['Tahun'].min()), int(df['Tahun'].max())],
    tahun_forecast = FORECAST_YRS,
    n_kecamatan    = int(df['kecamatan'].nunique()),
    tanaman        = sorted(df['tanaman'].unique().tolist()),
    n_cluster      = K_OPTIMAL,
    interval_level = INTERVAL_LEVEL,
    bobot_skor     = dict(produksi=W_PROD, produktivitas=W_PROD_V, trend=W_GROW),
    libs = dict(
        python     = platform.python_version(),
        numpy      = np.__version__,
        pandas     = pd.__version__,
        sklearn    = __import__('sklearn').__version__,
        statsmodels= __import__('statsmodels').__version__,
        scipy      = __import__('scipy').__version__,
    )
)

bundle = dict(
    meta               = meta,
    # Clustering
    scaler             = scaler,
    kmeans             = km_final,
    cluster_labels     = CLUSTER_LABELS,
    cluster_assignments= result_cl,
    # Time series
    selected_models    = selected_df,
    interval_df        = interval_df,
    forecast_long      = final_ts_df,
    forecast_kabupaten = forecast_kab,
    cv_wape            = wape_summary,
    cv_log             = cv_log,
    # Rekomendasi
    recommendations    = rekomendasi,
    rec_scores         = agg_rek_valid,
)

PKL_PATH = OUTPUT_DIR + 'agridash_model_bundle.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(bundle, f, protocol=4)

file_size = os.path.getsize(PKL_PATH) / 1024
print(f"✅ Bundle tersimpan  : {os.path.abspath(PKL_PATH)}")
print(f"   Ukuran file       : {file_size:.1f} KB")
print(f"   Run ID            : {RUN_ID}")
print(f"   Data SHA-256      : {data_hash[:16]}...")

In [ ]:
# ── 4.12 Export EWS CSV + Update Bundle ──────────────────────
# CSV
ews_kecamatan.to_csv(OUTPUT_DIR + 'output_ews_kecamatan.csv', index=False)
dashboard_summary.to_csv(OUTPUT_DIR + 'output_dashboard_summary.csv', index=False)
print("✅ output_ews_kecamatan.csv")
print("✅ output_dashboard_summary.csv")

# Update bundle — tambah dua tabel baru
import pickle as _pkl
with open(PKL_PATH, 'rb') as f:
    bundle = _pkl.load(f)

bundle['ews_kecamatan']    = ews_kecamatan      # per kec×tan×tahun_fc: status EWS
bundle['dashboard_summary'] = dashboard_summary  # per tan×tahun_fc: data kartu UI

with open(PKL_PATH, 'wb') as f:
    _pkl.dump(bundle, f, protocol=4)

file_size = os.path.getsize(PKL_PATH) / 1024
print(f"\n✅ Bundle diperbarui: {os.path.abspath(PKL_PATH)}")
print(f"   Ukuran            : {file_size:.1f} KB")
print(f"   Kunci bundle      : {sorted(bundle.keys())}")
print()
print("── Cara pakai di API backend ─────────────────────────────")
print("import pickle")
print("b = pickle.load(open('agridash_model_bundle.pkl','rb'))")
print()
print("# Kartu header tiap tanaman di dropdown:")
print("b['dashboard_summary'][b['dashboard_summary']['tanaman']=='Padi']")
print()
print("# Grafik aktual+forecast per tanaman (level kabupaten):")
print("b['forecast_kabupaten'][b['forecast_kabupaten']['tanaman']=='Padi']")
print()
print("# Status EWS tiap kecamatan untuk tanaman tertentu:")
print("b['ews_kecamatan'][b['ews_kecamatan']['tanaman']=='Padi']")
print()
print("# Forecast detail per kecamatan:")
print("b['forecast_long'][(b['forecast_long']['kecamatan']=='Pare') &")
print("                   (b['forecast_long']['tanaman']=='Padi')]")

In [ ]:
# ── 6.4 Verifikasi Bundle (Smoke Test) ───────────────────────
print("=" * 55)
print("  VERIFIKASI BUNDLE")
print("=" * 55)

with open(PKL_PATH, 'rb') as f:
    b = pickle.load(f)

checks = [
    ('meta keys',           sorted(b['meta'].keys())),
    ('tanaman di bundle',   b['meta']['tanaman']),
    ('n_cluster',           b['meta']['n_cluster']),
    ('forecast_long shape', b['forecast_long'].shape),
    ('forecast_kab shape',  b['forecast_kabupaten'].shape),
    ('rec shape',           b['recommendations'].shape),
    ('model_version',       b['meta']['model_version']),
    ('created_utc',         b['meta']['created_utc']),
]
for label, val in checks:
    print(f"  {label:<25}: {val}")

print()
# Test query sederhana
tan_test = b['meta']['tanaman'][0]
kec_test = b['cluster_assignments']['kecamatan'].iloc[0]
fc_test  = b['forecast_long'][
    (b['forecast_long']['kecamatan']==kec_test) &
    (b['forecast_long']['tanaman']==tan_test) &
    (b['forecast_long']['tipe']=='forecast')
][['tahun','produksi_ton','produksi_lower','produksi_upper']]

print(f"  Contoh query [{kec_test} × {tan_test}]:")
print(fc_test.to_string(index=False))
print()
print("✅ Bundle valid dan siap di-deploy")

In [ ]:
# ── 6.5 Manifest JSON (opsional, untuk verifikasi di server) ──
import glob

manifest_entries = []
for filepath in sorted(glob.glob(OUTPUT_DIR + '**', recursive=True)):
    if not os.path.isfile(filepath):
        continue
    size    = os.path.getsize(filepath)
    sha256  = hashlib.sha256(open(filepath,'rb').read()).hexdigest()
    manifest_entries.append(dict(
        file   = os.path.basename(filepath),
        size_kb= round(size/1024, 1),
        sha256 = sha256
    ))

manifest = dict(meta=meta, files=manifest_entries)
manifest_path = OUTPUT_DIR + 'manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)

print("=" * 55)
print("  OUTPUT FILES")
print("=" * 55)
for e in manifest_entries:
    icon = "🤖" if e['file'].endswith('.pkl') else "📊" if e['file'].endswith('.csv') else "🖼️" if e['file'].endswith('.png') else "📋"
    print(f"  {icon}  {e['file']:<45}  {e['size_kb']:>7.1f} KB")
print()
print(f"✅ manifest.json tersimpan: {manifest_path}")

## 7. Ringkasan

In [ ]:
print("=" * 65)
print("   RINGKASAN PEMODELAN — AGRIDASH KABUPATEN KEDIRI")
print("=" * 65)
print(f"\n[DATASET]")
print(f"   Periode     : {df['Tahun'].min()}–{df['Tahun'].max()} ({df['Tahun'].nunique()} tahun)")
print(f"   Tanaman     : {len(tanaman_list)} jenis | Kecamatan: {len(kecamatan_list)}")
print(f"   Baris       : {len(df):,}")

print(f"\n[CLUSTERING]")
print(f"   Fitur       : log1p(produksi) + proporsi luas panen per tanaman")
print(f"   Model utama : K-Means  k={K_OPTIMAL}  |  Silhouette: {sil_km:.4f}  |  DB: {db_km:.4f}")
print(f"   Pembanding  : Agglomerative         |  Silhouette: {sil_agg:.4f}  |  DB: {db_agg:.4f}")
print(f"   Auto-label  : berdasarkan total produksi centroid")

print(f"\n[TIME SERIES FORECAST]")
print(f"   Model kandidat : {', '.join(CANDIDATES)}")
print(f"   Seleksi        : WAPE pooled per tanaman (margin {MARGIN*100:.0f}%)")
for _, row in selected_df.iterrows():
    print(f"   {row['tanaman']:<20} {row['target']:<30} → {row['model']:>10}  WAPE={row['wape']:.3f}")
print(f"   Interval       : empiris {int(INTERVAL_LEVEL*100)}% (rasio aktual/prediksi CV)")
print(f"   Forecast       : {FORECAST_YRS}")

print(f"\n[REKOMENDASI TANAMAN]")
print(f"   Metode      : Skor Potensi Historis (OLS growth, norm per-tanaman)")
print(f"   Bobot       : Produksi {int(W_PROD*100)}% + Produktivitas {int(W_PROD_V*100)}% + Trend {int(W_GROW*100)}%")
print(f"   Top-{TOP_N} per kecamatan tersimpan")

print(f"\n[OUTPUT DEPLOY]")
print(f"   📦 agridash_model_bundle.pkl  ← LOAD INI DI API BACKEND")
print(f"   📋 manifest.json              ← verifikasi checksum SHA-256")
print(f"   📊 *.csv                      ← alternatif flat-file")
print("=" * 65)
print("   ✅ Notebook selesai — model siap di-deploy")
print("=" * 65)